# 00 - Setup check

Verifies the `ai-journey` environment: library versions, PyTorch + CUDA, and a GPU vs CPU matrix multiplication benchmark.

In [1]:
import sys, importlib

print(f"Python      {sys.version.split()[0]}  ({sys.executable})")
libs = ["numpy", "pandas", "matplotlib", "seaborn", "scipy", "sklearn", "xgboost",
        "lightgbm", "statsmodels", "jupyterlab", "ipykernel", "ipywidgets", "tqdm",
        "kaggle", "torch", "torchvision", "torchaudio", "ISLP"]
for name in libs:
    try:
        mod = importlib.import_module(name)
        print(f"{name:<12}{getattr(mod, '__version__', '?')}")
    except Exception as e:
        print(f"{name:<12}NOT AVAILABLE ({type(e).__name__}: {e})")

Python      3.13.7  (c:\Users\GIGABYTE\ai-journey\.venv\Scripts\python.exe)
numpy       2.5.5
pandas      2.3.3
matplotlib  3.11.2
seaborn     0.13.2
scipy       1.16.3
sklearn     1.9.1
xgboost     3.4.1
lightgbm    4.7.0
statsmodels 0.15.0
jupyterlab  4.6.6
ipykernel   7.4.0
ipywidgets  8.1.9
tqdm        4.70.1
Authentication required to call the Kaggle API.

First, you will need a Kaggle account. You can sign up at
  https://www.kaggle.com/account/login

Recommended: log in with OAuth via a web-based authorization flow.
No token to manage; credentials are cached locally for you.
    kaggle auth login

If you'd rather not use OAuth, generate an API token at
  https://www.kaggle.com/settings/api  (click "Generate New Token" under "API")
and supply it to the CLI in one of these ways:

  Option A: Environment variable
    export KAGGLE_API_TOKEN=xxxxxxxxxxxxxx  # token copied from the settings UI

  Option B: API token file
    Save the token to ~/.kaggle/access_token
kaggle      2.2.5


In [2]:
import time, torch

print("torch.cuda.is_available():", torch.cuda.is_available())
print("CUDA build:", torch.version.cuda)
N = 4096
a_cpu, b_cpu = torch.randn(N, N), torch.randn(N, N)

t0 = time.perf_counter()
c_cpu = a_cpu @ b_cpu
cpu_s = time.perf_counter() - t0
print(f"CPU  {N}x{N} matmul: {cpu_s*1000:8.1f} ms")

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    a_gpu, b_gpu = a_cpu.cuda(), b_cpu.cuda()
    (a_gpu @ b_gpu); torch.cuda.synchronize()   # warm-up
    t0 = time.perf_counter()
    c_gpu = a_gpu @ b_gpu
    torch.cuda.synchronize()
    gpu_s = time.perf_counter() - t0
    print(f"GPU  {N}x{N} matmul: {gpu_s*1000:8.1f} ms")
    print(f"Speed-up: {cpu_s/gpu_s:.1f}x")
    print("Max abs diff CPU vs GPU:", (c_gpu.cpu() - c_cpu).abs().max().item())
else:
    print("No CUDA GPU available - CPU only.")

torch.cuda.is_available(): True
CUDA build: 12.8
CPU  4096x4096 matmul:    374.9 ms
GPU: NVIDIA GeForce RTX 4060 Laptop GPU
GPU  4096x4096 matmul:     18.3 ms
Speed-up: 20.5x
Max abs diff CPU vs GPU: 0.001007080078125
